# 01 — PyTorch Introduction: Tensors and Neural Networks

**PyTorch** is the most popular deep learning framework for research and production.

Deep learning = machine learning with **neural networks** — layers of mathematical operations that learn patterns from data.

You'll learn:
- What a **tensor** is (the basic building block)
- How PyTorch computes **gradients** automatically (the magic behind learning)
- How to build and train a simple neural network
- How to use **GPU/MPS acceleration** if available

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt

# Detect the best available device
if torch.cuda.is_available():
    device = 'cuda'   # Nvidia GPU (Windows)
elif torch.backends.mps.is_available():
    device = 'mps'    # Apple Silicon GPU (Mac M1/M2/M3)
else:
    device = 'cpu'

print(f'PyTorch version: {torch.__version__}')
print(f'Using device: {device}')

## Part 1: Tensors

A **tensor** is just an n-dimensional array — like a numpy array but with GPU support and automatic differentiation.

In [ ]:
# Create tensors
scalar = torch.tensor(3.14)
vector = torch.tensor([1.0, 2.0, 3.0])
matrix = torch.tensor([[1, 2], [3, 4]], dtype=torch.float32)

print('Scalar:', scalar, '| Shape:', scalar.shape)
print('Vector:', vector, '| Shape:', vector.shape)
print('Matrix:')
print(matrix, '| Shape:', matrix.shape)

In [ ]:
# Tensor operations — look familiar if you've used numpy
a = torch.tensor([1.0, 2.0, 3.0])
b = torch.tensor([4.0, 5.0, 6.0])

print('Add:     ', a + b)
print('Multiply:', a * b)
print('Dot:     ', torch.dot(a, b))
print('Mean:    ', a.mean())

# Random tensors (used to initialize neural network weights)
rand_matrix = torch.randn(3, 4)   # 3x4 matrix of random normal values
print('\nRandom 3x4 matrix:')
print(rand_matrix)

## Part 2: Automatic Differentiation (autograd)

This is what makes deep learning work. PyTorch can automatically compute derivatives (gradients), which is what gradient descent uses to update model weights.

In [ ]:
# requires_grad=True tells PyTorch to track operations on this tensor
x = torch.tensor(3.0, requires_grad=True)

# Define a function: y = x^2 + 2x + 1
y = x**2 + 2*x + 1
print('y =', y.item())

# Compute gradients: dy/dx = 2x + 2
y.backward()
print('dy/dx at x=3:', x.grad.item())  # Should be 2*3 + 2 = 8

## Part 3: Building a Neural Network

We'll build a network to classify the Iris dataset (3 species from 4 features).

Architecture: `4 inputs → 16 neurons → 8 neurons → 3 outputs`

In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Load and prepare data
iris = load_iris()
X, y = iris.data, iris.target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

# Convert to PyTorch tensors
X_train_t = torch.tensor(X_train, dtype=torch.float32).to(device)
X_test_t  = torch.tensor(X_test,  dtype=torch.float32).to(device)
y_train_t = torch.tensor(y_train, dtype=torch.long).to(device)
y_test_t  = torch.tensor(y_test,  dtype=torch.long).to(device)

print('Data ready!')

In [ ]:
# Define the neural network
class IrisNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(4, 16),    # input layer: 4 features → 16 neurons
            nn.ReLU(),           # activation function (adds non-linearity)
            nn.Linear(16, 8),   # hidden layer: 16 → 8
            nn.ReLU(),
            nn.Linear(8, 3),    # output layer: 8 → 3 classes
        )

    def forward(self, x):
        return self.network(x)

model = IrisNet().to(device)
print(model)
total_params = sum(p.numel() for p in model.parameters())
print(f'\nTotal parameters: {total_params}')

In [ ]:
# Training setup
criterion = nn.CrossEntropyLoss()         # loss for multi-class classification
optimizer = optim.Adam(model.parameters(), lr=0.01)  # Adam adjusts learning rate automatically

# Training loop
losses = []
n_epochs = 200

for epoch in range(n_epochs):
    model.train()
    optimizer.zero_grad()           # clear gradients from previous step
    outputs = model(X_train_t)      # forward pass
    loss = criterion(outputs, y_train_t)  # compute loss
    loss.backward()                 # compute gradients
    optimizer.step()                # update weights
    losses.append(loss.item())

    if (epoch + 1) % 50 == 0:
        print(f'Epoch {epoch+1}/{n_epochs}, Loss: {loss.item():.4f}')

In [ ]:
# Plot training loss
plt.figure(figsize=(8, 4))
plt.plot(losses)
plt.title('Training Loss over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Evaluate on test set
model.eval()
with torch.no_grad():  # no need to compute gradients during evaluation
    outputs = model(X_test_t)
    _, predicted = torch.max(outputs, 1)
    accuracy = (predicted == y_test_t).float().mean()

print(f'Test Accuracy: {accuracy.item():.1%}')
print(f'Predicted: {predicted.cpu().numpy()}')
print(f'Actual:    {y_test}')

## Summary

You've built and trained a real neural network! The training loop pattern is:

```
for each epoch:
    zero_grad() → forward() → compute loss → backward() → step()
```

**Try:** Change `lr=0.01` to `0.001` or `0.1`. What happens to training speed and accuracy?

**Next:** `03_computer_vision/01_image_basics.ipynb` — apply PyTorch to images!